# Import

In [ ]:
import numpy as np
import healpy as hp
import pandas as pd
import networkx as nx
import geopandas as gpd

from itertools import combinations

from pyproj import Geod
from shapely import union_all, intersects
from shapely.geometry import LineString
from healpix_geo.nested import healpix_to_lonlat
from healpix_geo.nested import lonlat_to_healpix

import plotly.express as px
import plotly.graph_objects as go

import matplotlib.pyplot as plt

In [ ]:
def get_points(NSIDE):
    NPIX = hp.nside2npix(NSIDE)
    points = np.array(
        hp.pix2vec(NSIDE, range(NPIX), nest=True)
    ).T

    df = pd.DataFrame(points, columns=('x', 'y', 'z'))

    return df

def get_patches(NSIDE):
    NPIX = hp.nside2npix(NSIDE)

    pixels = np.arange(NPIX)
    patches = np.array(
        [hp.boundaries(NSIDE, i, nest=True).T for i in pixels]
    ).reshape(-1, 3)

    df = pd.DataFrame(patches, columns=('x', 'y', 'z'))

    return df

# Data

## Borders

In [ ]:
kaz = gpd.read_file('data/geoBoundaries-KAZ-ADM0_simplified.geojson')
ukr = gpd.read_file('data/geoBoundaries-UKR-ADM0_simplified.geojson')
rus = gpd.read_file('data/geoBoundaries-RUS-ADM0_simplified.geojson')
blr = gpd.read_file('data/geoBoundaries-BLR-ADM0_simplified.geojson')
borders = pd.concat((kaz, ukr, rus, blr)).reset_index(drop=True)
borders = borders.to_crs(epsg=4326)
borders

## Airports

In [ ]:
airports = pd.read_csv('data/airports.csv')
airports = gpd.GeoDataFrame(
    airports.drop(['GeoPointLat', 'GeoPointLong'], axis=1),
    geometry=gpd.points_from_xy(
        *airports[['GeoPointLong', 'GeoPointLat']].values.T
    ), crs='EPSG:4326'
)
airports[airports.ICAO.isin(['KJFK', 'UAAA'])]

# Maps

In [ ]:
fig = px.choropleth(
    borders,
    geojson=borders.geometry,
    locations=borders.shapeISO,
    locationmode='ISO-3',
    color='shapeName',
)
fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showcoastlines=True,
        showland=True,
        #showcountries=True,
    ),
)
fig.show()

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=px.colors.qualitative.Plotly[:len(countries)],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(route)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showcoastlines=True,
        showland=True,
        showcountries=True,
    ),
)
fig.show()

# HEALPix

In [ ]:
NSIDE = 16
healpix = get_points(NSIDE)

NPIX = hp.nside2npix(NSIDE)
ipix = np.arange(NPIX)
depth = np.log2(NSIDE).astype(int)

print(NPIX, depth)

In [ ]:
lon, lat = healpix_to_lonlat(ipix, depth=depth, ellipsoid='WGS84')
lon = (lon + 180) % 360 - 180

In [ ]:
gdf = gpd.GeoDataFrame(
    healpix,
    geometry=gpd.points_from_xy(
        lon, lat
    ),
    crs='EPSG:4326',
)
gdf

In [ ]:
gdf['avoid'] = np.array([
    gdf.geometry.within(shape)
    for _, shape
    in borders[borders.shapeISO!='KAZ'].geometry.items()
]).sum(axis=0)

avoid_neighbours = hp.get_all_neighbours(
    NSIDE,
    gdf[gdf.avoid==1].index.values,
    nest=True,
)
avoid_neighbours[np.where(avoid_neighbours<0)] = np.where(avoid_neighbours<0)[0]
gdf.loc[np.unique(avoid_neighbours), 'avoid'] = 1

obstacle_mtx = 1 - np.diag(gdf.avoid)

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=px.colors.qualitative.Plotly[:len(countries)],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

points = go.Scattergeo(
    lon=gdf[gdf.avoid!=1].geometry.x,
    lat=gdf[gdf.avoid!=1].geometry.y,
    mode='markers',
    marker=dict(size=2, color='black'),
)
fig.add_trace(points)

route = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(route)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()

# Graph

In [ ]:
geod = Geod(ellps='WGS84')

shape_restricted = union_all(
    borders.loc[1:3, 'geometry']
)

start = lonlat_to_healpix(
    airports[airports.ICAO=='UAAA'].geometry.x,
    airports[airports.ICAO=='UAAA'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]
end = lonlat_to_healpix(
    airports[airports.ICAO=='KJFK'].geometry.x,
    airports[airports.ICAO=='KJFK'].geometry.y,
    depth,
    ellipsoid='WGS84',
)[0]

def great_circle_path(a=None, b=None, n_points=100, iterable=False):
    if iterable:
        (a, b) = a

    points = geod.npts(
        a.x, a.y,
        b.x, b.y,
        n_points,
    )
    line = LineString(points)

    return line

def great_circle_distance(a=None, b=None, iterable=False):
    if iterable:
        (a, b) = a

    distance = geod.inv(
        a.x, a.y,
        b.x, b.y,
    )[2]

    return distance

In [ ]:
point_pairs = pd.DataFrame(
    combinations(gdf[gdf.avoid==0].geometry, 2)
)

In [ ]:
point_pairs['path'] = point_pairs.apply(
    great_circle_path,
    axis=1,
    raw=True,
    iterable=True,
    n_points=50,
)

In [ ]:
point_pairs['intersect_avoid'] = gpd.GeoSeries(
    point_pairs.path
).intersects(shape_restricted)

In [ ]:
dim = len(gdf.index)
connection_mtx = np.diag(np.zeros(dim))

import concurrent.futures:
for idx, row in enumerate(connection_mtx):
    points_idx = np.arange(len(row))
    points_idx = points_idx[points_idx!=idx]

    points = gdf.iloc[points_idx].geometry

    paths = [
        great_circle_path(
            gdf.iloc[idx],
            gdf.iloc[point],
        )
        for point in points_idx
    ]
    distances = [
        0
        if path.intersects(shape_restricted)
        else great_circle_distance(
            gdf.iloc[idx],
            gdf.iloc[point],
        )
        for point, path in zip(points_idx, paths)
    ]

    connection_mtx[idx, points_idx] = distances

connection_mtx[gdf[gdf.avoid==1].index] = 0

In [ ]:
G = nx.from_numpy_array(connection_mtx)

In [ ]:
route = nx.astar_path(
    G, start, end,
)

In [ ]:
countries = ['KAZ', 'UKR', 'RUS', 'BLR', 'USA']

fig = go.Figure()

trace = go.Choropleth(
    locations=countries,
    z=list(range(len(countries))),
    locationmode='ISO-3',
    colorscale=px.colors.qualitative.Plotly[:len(countries)],
    text=countries,
    showscale=False,
)
fig.add_trace(trace)

route_trace = go.Scattergeo(
    lon=gdf.iloc[route].geometry.x,
    lat=gdf.iloc[route].geometry.y,
    mode='lines',
    marker=dict(size=1, color='blue'),
)
fig.add_trace(route_trace)

start_end = go.Scattergeo(
    lon=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.x,
    lat=airports[airports.ICAO.isin(['KJFK', 'UAAA'])].geometry.y,
    mode='lines+markers',
    marker=dict(size=5, color='red'),
)
fig.add_trace(start_end)

fig.update_geos(
    fitbounds='locations',
    visible=False,
    projection_type='orthographic',
)
fig.update_layout(
    autosize=False,
    width=900, height=600,
    margin=dict(l=0, r=0, b=0, t=0),
    geo=dict(
        showframe=True,
        showland=True,
    ),
)
fig.show()